kernel : Python (Pixi)

In [ ]:
import anndata
import gc
import os
from anndata import AnnData
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir

anndata.settings.allow_write_nullable_strings = True

raw_count_matrix_location = find_env_dir("RAW_COUNT_MATRIX")
h5ad_matrix_location = find_env_dir("H5AD_MATRIX")

def load_rds_data(rds_path: str, gz: bool, series: str) -> AnnData:
    import anndata2ri
    from rpy2.robjects import r
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects.packages import importr

    importr("base")
    importr("Matrix")
    importr("Seurat")
    importr("SingleCellExperiment")
    importr("monocle3")
    
    if gz:
        read_cmd = f'readRDS(gzcon(gzfile("{rds_path}", "rb")))'
    else:
        read_cmd = f'readRDS("{rds_path}")'

    rcode = f"""
    obj <- {read_cmd}
    if (inherits(obj, "Seurat")) {{
        obj <- as.SingleCellExperiment(obj)
    }}

    if (!inherits(obj, "SingleCellExperiment")) {{
        stop("Loaded object is not a SingleCellExperiment (or derived from it).")
    }}

    anames <- assayNames(obj)
    use_assay <- if ("counts" %in% anames) "counts" else anames[[1]]

    mat <- assay(obj, use_assay)
    mat <- as(mat, "dgCMatrix")

    assay(obj, "counts") <- mat
    # Overwrite 'X' assay with 'counts' assay
    assay(obj, "X") <- assay(obj, "counts")
    obj <- as(obj, "SingleCellExperiment") # Ensure the object is a SingleCellExperiment
    obj
    """
    with localconverter(anndata2ri.converter):
        adata = r(rcode)
        r("if (exists('obj')) rm(obj)")
        r("gc()")

    if not isinstance(adata, AnnData):
        raise ValueError(f"Result is not AnnData. Got type: {type(adata)}")

    adata.layers.pop("counts", None)
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)
    
    # Multidimensional representations (obsm, varm, layers) are intentionally cleared at this stage,
    # since integrated multi-modal / multi-dimensional analyses will be performed after dataset integration, 
    # overwriting any existing representations.
    adata.obsm = {}
    adata.varm = {}
    adata.layers = {}
    gc.collect()

    adata.obs["series"] = series
    adata.obs["batch"] = "not set"
    return adata

In [27]:
FILE = "GSE249268_Final_aging_OPC_cds_mm106.rds.gz"
SERIES_NAME = "Heo"
SAVE_FILE_NAME = "Heo.h5ad"
adata = load_rds_data(os.path.join(raw_count_matrix_location, SERIES_NAME, FILE), gz = True, series = SERIES_NAME)
adata.obs.head() #type: ignore

,barcode,Size_Factor,sample_id,Total_UMIs,num_genes_expressed,batch,processing_date,age,sex,mt_reads,total_reads,mito_ratio,Total_mRNA,cluster,identity,new_age,new_identity,final_identity,series
AAACCCAGTCATTGCA.P30M-1,AAACCCAGTCATTGCA,0.616237,P30M-1,3832.0,2151,not set,20190509,30,M,148.0,3832.0,0.038622,3832.0,24,quiescent_OPC,1_P30,1_quiescent_OPC,quiescent_OPC,Heo
AAACCCAGTTTCACAG.P30M-1,AAACCCAGTTTCACAG,1.184551,P30M-1,7369.0,3400,not set,20190509,30,M,303.0,7366.0,0.041135,7366.0,21,quiescent_OPC,1_P30,1_quiescent_OPC,transitioning_OPC,Heo
AAACGAAAGCAAGGAA.P30M-1,AAACGAAAGCAAGGAA,0.981444,P30M-1,6104.0,3095,not set,20190509,30,M,232.0,6103.0,0.038014,6103.0,6,quiescent_OPC,1_P30,1_quiescent_OPC,quiescent_OPC,Heo
AAACGCTCAAGGCCTC.P30M-1,AAACGCTCAAGGCCTC,1.097069,P30M-1,6824.0,3487,not set,20190509,30,M,135.0,6822.0,0.019789,6822.0,14,quiescent_OPC,1_P30,1_quiescent_OPC,quiescent_OPC,Heo
AAACGCTGTCCCACGA.P30M-1,AAACGCTGTCCCACGA,0.842662,P30M-1,5240.0,2787,not set,20190509,30,M,191.0,5240.0,0.036450,5240.0,6,quiescent_OPC,1_P30,1_quiescent_OPC,quiescent_OPC,Heo


In [31]:
adata.obs["age"].value_counts()

age
30     16260
720     8967
360     8332
180     5248
Name: count, dtype: int64

In [32]:
adata.obs["celltype"] = adata.obs["final_identity"]
adata.obs["sex"] = adata.obs["sex"]
adata.obs["age"] = adata.obs["age"]
adata.obs["sample"] = adata.obs["sample_id"]
adata.obs["series"] = adata.obs["series"]
keep = ["celltype", "sex", "age", "sample", "series", "batch"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [ ]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(h5ad_matrix_location, SAVE_FILE_NAME))
del adata
gc.collect()